# ML-09 — Validation and Research Claim Audit

This notebook audits my Week-5 Lane II model rather than changing the project halfway through. I keep the same target, features, and Logistic Regression, then compare a deliberately naive row-random split with the client-grouped holdout used in Week 5.

The goal here is not to chase the biggest number. It is to check whether the validation design supports the claim I want to make, look for leakage one more time, and keep the final interpretation at a level the evidence can honestly carry.

> Skill focus: `hunting-leakage-and-validating` + careful research-claim language.

## 1. Two paper findings + my methodology questions

I used two findings from the FlyRank reference analysis supplied with the internship as concrete examples for this audit.

### Finding 1 — the learned ranking beat the hand-written baseline

The FlyRank reference report shows a Random Forest reaching **Precision@50 = 0.740**, compared with **0.240** for the rule baseline, under a `client_holdout` validation split.

- **Where does the label come from?** The target is `is_declining_label`, an observed decline proxy derived from the dataset's trend fields.
- **Methodology question:** Does a client holdout support the claim that the ranking generalizes beyond the clients used for training?
- **My reading:** It is stronger than a row-random split because pages from the same client are not shared across train and test. However, it still evaluates an observed snapshot label; it does not by itself prove future decline or that refreshing content will cause recovery.

### Finding 2 — several search/content signals were important to the model

The reference report lists signals such as days with impressions, impression volume, average position, and content age among its leading model features.

- **Where does the label come from?** The same observed decline proxy is used.
- **Methodology question:** Does feature importance show that these signals *cause* search decline?
- **My reading:** No. Feature importance shows how the fitted model used signals to separate the observed labels. It supports an association or decision-support interpretation, not a causal statement about Google's ranking system.

These two examples set the standard I use below for auditing my own Week-5 result.

In [1]:
import pandas as pd
from IPython.display import display

paper_audit = pd.DataFrame([
    {
        "finding": "Learned ranking > rule baseline",
        "reported_evidence": "Precision@50: 0.740 vs 0.240",
        "label_source": "Observed is_declining_label",
        "validation": "Client holdout",
        "claim_that_holds": "Out-of-group ranking evidence on the observed label"
    },
    {
        "finding": "Search/content signals rank as important",
        "reported_evidence": "Model feature importance",
        "label_source": "Observed is_declining_label",
        "validation": "Model evaluated on held-out clients",
        "claim_that_holds": "Association / decision-support, not causation"
    }
])

display(paper_audit)


,finding,reported_evidence,label_source,validation,claim_that_holds
0,Learned ranking > rule baseline,Precision@50: 0.740 vs 0.240,Observed is_declining_label,Client holdout,Out-of-group ranking evidence on the observed ...
1,Search/content signals rank as important,Model feature importance,Observed is_declining_label,Model evaluated on held-out clients,"Association / decision-support, not causation"


## 2. My model under an honest split (before/after)

My Week-5 notebook already used an **80/20 grouped holdout by `client_id`**, so I do not want to pretend Week 6 is fixing a split that was already grouped.

Instead, I use a **row-random split as a deliberately naive comparison** and then re-run the exact same Logistic Regression under the Week-5 client-grouped holdout.

The comparison asks two different questions:

- **Row-random split:** can the model rank new pages when the same clients can appear in both training and test data?
- **Client-grouped split:** can the model rank pages from clients it did not train on?

I treat **Precision@50** as the primary operational metric because the Lane II output is a short review queue. I also report the test base rate, lift over that base rate, and ROC-AUC for context.

In [2]:
# ---------- Colab setup + Week-5 model under two validation designs ----------

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# Make the notebook runnable directly in Google Colab.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/pretom26/ml_internship_flyrankAI.git"
REPO_DIR = Path("/content/ml_internship_flyrankAI")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True
        )
    os.chdir(REPO_DIR)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run this notebook from the repo or let the Colab setup clone it."
    )

# Same starter data and same observed label used in Week 5.
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(np.int8)
)

# Same Week-5 handling: avg_position == 0 means "no data", not rank zero.
df["avg_position_no_data"] = (
    pd.to_numeric(df["avg_position"], errors="coerce").fillna(0).eq(0)
).astype(np.int8)

df["avg_position_model"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
).replace(0, np.nan)

for col in ["days_since_last_update", "impressions_90d", "ctr", "trend_pct"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

FEATURES_NUMERIC = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position_model",
    "avg_position_no_data",
]
FEATURES_CATEGORICAL = ["content_type"]
FINAL_FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL
TARGET = "is_declining_label"

def make_model(numeric_features=None):
    numeric_features = numeric_features or FEATURES_NUMERIC

    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, FEATURES_CATEGORICAL),
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("logreg", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_true[order].mean())

def evaluate_split(split_name, train_idx, test_idx):
    train_part = df.iloc[train_idx].copy()
    test_part = df.iloc[test_idx].copy()

    model = make_model()
    model.fit(train_part[FINAL_FEATURES], train_part[TARGET])

    y_test = test_part[TARGET].to_numpy()
    scores = model.predict_proba(test_part[FINAL_FEATURES])[:, 1]

    train_clients = set(train_part["client_id"].astype(str))
    test_clients = set(test_part["client_id"].astype(str))
    overlap = len(train_clients & test_clients)

    base_rate = float(y_test.mean())
    p20 = precision_at_k(y_test, scores, 20)
    p50 = precision_at_k(y_test, scores, 50)
    p100 = precision_at_k(y_test, scores, 100)

    metrics = {
        "split": split_name,
        "train_rows": len(train_part),
        "test_rows": len(test_part),
        "client_overlap": overlap,
        "test_base_rate": base_rate,
        "Precision@20": p20,
        "Precision@50": p50,
        "Precision@100": p100,
        "Lift@50": p50 / base_rate if base_rate > 0 else np.nan,
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Average Precision": average_precision_score(y_test, scores),
    }

    return metrics, model, scores

# 1) Deliberately naive row-random split.
all_idx = np.arange(len(df))
random_train_idx, random_test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=df[TARGET],
)

# 2) Honest Week-5 split: clients never cross train/test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
group_train_idx, group_test_idx = next(
    gss.split(df, y=df[TARGET], groups=df["client_id"])
)

random_metrics, random_model, random_scores = evaluate_split(
    "Naive row-random", random_train_idx, random_test_idx
)
group_metrics, group_model, group_scores = evaluate_split(
    "Client-grouped holdout", group_train_idx, group_test_idx
)

results_df = pd.DataFrame([random_metrics, group_metrics])

print("WEEK-5 MODEL: VALIDATION BEFORE / AFTER")
display(
    results_df[
        [
            "split", "train_rows", "test_rows", "client_overlap",
            "test_base_rate", "Precision@20", "Precision@50",
            "Precision@100", "Lift@50", "ROC-AUC", "Average Precision"
        ]
    ].round(3)
)

random_p50 = random_metrics["Precision@50"]
group_p50 = group_metrics["Precision@50"]
gap = random_p50 - group_p50

print(f"Naive row-random Precision@50:      {random_p50:.3f}")
print(f"Client-grouped Precision@50:        {group_p50:.3f}")
print(f"Difference (random - grouped):      {gap:+.3f}")
print(f"Grouped test decline base rate:     {group_metrics['test_base_rate']:.3f}")
print(f"Grouped Lift@50 over base rate:     {group_metrics['Lift@50']:.2f}x")
print(f"Grouped client overlap:             {group_metrics['client_overlap']}")

if gap > 0:
    print(
        "\nInterpretation: the row-random split looks stronger, but it allows the same "
        "clients on both sides. I therefore keep the lower/more demanding client-grouped "
        "result as the number I trust for research claims."
    )
elif gap < 0:
    print(
        "\nInterpretation: the grouped result is not lower in this run, but it is still "
        "the more relevant test for unseen-client generalization because client overlap is zero."
    )
else:
    print(
        "\nInterpretation: both splits give the same Precision@50 in this run. I still "
        "prefer the grouped result because it tests the harder unseen-client question."
    )


WEEK-5 MODEL: VALIDATION BEFORE / AFTER


,split,train_rows,test_rows,client_overlap,test_base_rate,Precision@20,Precision@50,Precision@100,Lift@50,ROC-AUC,Average Precision
0,Naive row-random,24000,6000,31,0.542,0.40,0.58,0.69,1.070,0.607,0.615
1,Client-grouped holdout,23837,6163,0,0.511,0.55,0.62,0.60,1.213,0.563,0.555


Naive row-random Precision@50:      0.580
Client-grouped Precision@50:        0.620
Difference (random - grouped):      -0.040
Grouped test decline base rate:     0.511
Grouped Lift@50 over base rate:     1.21x
Grouped client overlap:             0

Interpretation: the grouped result is not lower in this run, but it is still the more relevant test for unseen-client generalization because client overlap is zero.


## 3. Leakage audit

I re-check the exact feature set used above rather than assuming the Week-5 choices are safe.

My main leakage rules are:

1. `trend_direction`, `trend_pct`, and the target cannot be model inputs because the target is defined from the trend outcome.
2. `content_id` and `client_id` are identifiers. `client_id` is useful for splitting, but neither ID is a predictive feature.
3. Existing recommendation, priority, or model-output flags should not become features.
4. The starter dataset is a **single observed snapshot**. Its search metrics and decline label are contemporaneous, so this model can support ranking of an observed decline proxy, but I should **not describe this Week-6 result as future forecasting**.

As a sanity check, I deliberately add `trend_pct` to a throwaway model. If its score jumps sharply, that is evidence that my evaluation harness can detect a label-derived shortcut. The deliberately leaky model is never used for recommendations or final claims.

In [3]:
# ---------- Leakage audit on the final Week-5 feature set ----------

FORBIDDEN = {
    "is_declining_label",  # target itself
    "trend_direction",     # directly defines the target
    "trend_pct",           # sibling/source of trend_direction
    "content_id",          # identifier
    "client_id",           # grouping key only
}

overlap_with_forbidden = sorted(set(FINAL_FEATURES) & FORBIDDEN)
assert not overlap_with_forbidden, (
    f"Leakage detected in final features: {overlap_with_forbidden}"
)

# Product/recommendation-like fields are also suspicious if they ever appear.
suspicious_tokens = ("priority", "recommend", "opportunity", "refresh_score", "model_score")
product_like_columns = sorted([
    c for c in df.columns
    if any(token in c.lower() for token in suspicious_tokens)
])
product_like_used = sorted(set(product_like_columns) & set(FINAL_FEATURES))
assert not product_like_used, (
    f"Decision-derived feature(s) found: {product_like_used}"
)

feature_audit = pd.DataFrame([
    {
        "feature": "days_since_last_update",
        "role": "content freshness",
        "label_derived": False,
        "identifier": False,
        "timing_note": "observed at the same snapshot",
        "status": "KEEP"
    },
    {
        "feature": "impressions_90d",
        "role": "search demand",
        "label_derived": False,
        "identifier": False,
        "timing_note": "trailing metric at the same snapshot",
        "status": "KEEP, but not a future-proof claim"
    },
    {
        "feature": "ctr",
        "role": "search efficiency",
        "label_derived": False,
        "identifier": False,
        "timing_note": "trailing metric at the same snapshot",
        "status": "KEEP, but not a future-proof claim"
    },
    {
        "feature": "avg_position_model",
        "role": "search visibility",
        "label_derived": False,
        "identifier": False,
        "timing_note": "trailing metric at the same snapshot",
        "status": "KEEP, but not a future-proof claim"
    },
    {
        "feature": "avg_position_no_data",
        "role": "missing-position indicator",
        "label_derived": False,
        "identifier": False,
        "timing_note": "derived only from avg_position availability",
        "status": "KEEP"
    },
    {
        "feature": "content_type",
        "role": "content category",
        "label_derived": False,
        "identifier": False,
        "timing_note": "content metadata",
        "status": "KEEP"
    },
])

print("FINAL FEATURE LEAKAGE AUDIT")
display(feature_audit)

print("Forbidden features used:", overlap_with_forbidden)
print("Decision/product-like features used:", product_like_used)
print(
    "Temporal limitation: features and label come from the same starter snapshot, "
    "so this validates observed decline ranking, not future decline forecasting."
)

# Deliberate bad-model test: add trend_pct, which is label-derived.
# This model exists only to verify that the audit can expose a shortcut.
if "trend_pct" in df.columns:
    leaky_numeric = FEATURES_NUMERIC + ["trend_pct"]
    leaky_model = make_model(numeric_features=leaky_numeric)

    leaky_train = df.iloc[group_train_idx]
    leaky_test = df.iloc[group_test_idx]

    leaky_features = leaky_numeric + FEATURES_CATEGORICAL
    leaky_model.fit(leaky_train[leaky_features], leaky_train[TARGET])

    leaky_y = leaky_test[TARGET].to_numpy()
    leaky_scores = leaky_model.predict_proba(leaky_test[leaky_features])[:, 1]

    leaky_p50 = precision_at_k(leaky_y, leaky_scores, 50)
    leaky_auc = roc_auc_score(leaky_y, leaky_scores)

    print("\nDELIBERATE LEAKAGE SANITY CHECK — BAD MODEL, DO NOT USE")
    print(f"Honest feature set Precision@50: {group_metrics['Precision@50']:.3f}")
    print(f"+ trend_pct Precision@50:        {leaky_p50:.3f}")
    print(f"Honest feature set ROC-AUC:      {group_metrics['ROC-AUC']:.3f}")
    print(f"+ trend_pct ROC-AUC:             {leaky_auc:.3f}")

    if leaky_auc > group_metrics["ROC-AUC"] + 0.10:
        print(
            "Result: the label-derived feature creates a large artificial gain. "
            "That is exactly why trend_pct stays excluded."
        )
    else:
        print(
            "Result: the gain is smaller than expected, but trend_pct is still excluded "
            "because it is label-derived by definition."
        )


FINAL FEATURE LEAKAGE AUDIT


,feature,role,label_derived,identifier,timing_note,status
0,days_since_last_update,content freshness,False,False,observed at the same snapshot,KEEP
1,impressions_90d,search demand,False,False,trailing metric at the same snapshot,"KEEP, but not a future-proof claim"
2,ctr,search efficiency,False,False,trailing metric at the same snapshot,"KEEP, but not a future-proof claim"
3,avg_position_model,search visibility,False,False,trailing metric at the same snapshot,"KEEP, but not a future-proof claim"
4,avg_position_no_data,missing-position indicator,False,False,derived only from avg_position availability,KEEP
5,content_type,content category,False,False,content metadata,KEEP


Forbidden features used: []
Decision/product-like features used: []
Temporal limitation: features and label come from the same starter snapshot, so this validates observed decline ranking, not future decline forecasting.

DELIBERATE LEAKAGE SANITY CHECK — BAD MODEL, DO NOT USE
Honest feature set Precision@50: 0.620
+ trend_pct Precision@50:        1.000
Honest feature set ROC-AUC:      0.563
+ trend_pct ROC-AUC:             0.955
Result: the label-derived feature creates a large artificial gain. That is exactly why trend_pct stays excluded.


## 4. Claim rewrite

My boldest possible version would be:

> **Too strong:** “My model predicts which pages will decline and tells editors which pages they should refresh.”

That sentence goes beyond the evidence. This Week-6 model uses an observed snapshot decline proxy, not a future outcome, and there is no intervention showing that a refresh causes recovery.

My final wording therefore reports the held-out ranking result directly, states the population it was tested on, and describes the output as **decision-support for manual review** rather than an automatic or causal recommendation.

In [4]:
# ---------- Build the final claim from the honest held-out result ----------

honest = results_df.set_index("split").loc["Client-grouped holdout"]

safe_claim = (
    f"On the client-grouped holdout, the Logistic Regression model achieved "
    f"Precision@50 = {honest['Precision@50']:.3f} against a test decline base rate "
    f"of {honest['test_base_rate']:.3f}, equal to {honest['Lift@50']:.2f}x lift at "
    f"the top 50. In this anonymized starter dataset, the model therefore showed "
    f"directional value for prioritizing pages for manual review on the observed "
    f"decline proxy. This result does not prove that the included signals cause "
    f"decline, does not predict Google's algorithm, and does not show that refreshing "
    f"a flagged page will recover performance."
)

print("FINAL CLAIM")
print(safe_claim)


FINAL CLAIM
On the client-grouped holdout, the Logistic Regression model achieved Precision@50 = 0.620 against a test decline base rate of 0.511, equal to 1.21x lift at the top 50. In this anonymized starter dataset, the model therefore showed directional value for prioritizing pages for manual review on the observed decline proxy. This result does not prove that the included signals cause decline, does not predict Google's algorithm, and does not show that refreshing a flagged page will recover performance.


## Self-check

Before I submit, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (tick after **Runtime → Run all** in Colab)
- [x] No client names, URLs, or private queries are displayed
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — tick after saving the completed notebook to GitHub

**Final check before submission:** run all cells once from a fresh Colab runtime and make sure the printed grouped metrics match the numbers I quote anywhere else in the project.